# Evaluating a NIM-powered RAG agent with OpenGATE

This notebook builds a tiny retrieval-augmented (RAG) agent whose answers are generated by an **NVIDIA NIM** model, then checks each answer with **OpenGATE** — deterministically, with **no LLM judge**.

OpenGATE asks three questions of every answer:

1. **Are the required facts present?** (answer-anchor recall)
2. **Does every number trace back to the retrieved context?** (no fabrication)
3. **When the context can't answer, did the agent abstain** rather than invent something?

Because the check is pure logic, it's free, instant, and reproducible — you can run it on every answer, or gate it in CI. The same check ships as a Python package (`opengate-grounding`), an MCP server, a CLI, and a Docker image.

> **You'll need a free NVIDIA API key** from [build.nvidia.com](https://build.nvidia.com) — it exposes NIM models through an OpenAI-compatible API. Set it as `NVIDIA_API_KEY`. *(No key handy? The notebook falls back to canned answers so you can still see the evaluation mechanics end-to-end.)*

In [ ]:
%pip install --quiet openai opengate-grounding

## Configure the NIM client

NIM speaks the OpenAI API, so we use the `openai` client and just point its `base_url` at NVIDIA's endpoint. Set `NIM_BASE_URL` to a self-hosted NIM container to run entirely locally.

In [ ]:
import os

MODEL = os.environ.get('NIM_MODEL', 'meta/llama-3.1-8b-instruct')
BASE_URL = os.environ.get('NIM_BASE_URL', 'https://integrate.api.nvidia.com/v1')
API_KEY = os.environ.get('NVIDIA_API_KEY')

client = None
if API_KEY:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    print(f'Using NIM model {MODEL} at {BASE_URL}')
else:
    print('No NVIDIA_API_KEY set — using canned answers so the eval still runs. '
          'Set the key to evaluate a real NIM model.')

## 1. A tiny knowledge base

Four short generic product-docs passages. In a real system these come from your vector store or search index.

In [ ]:
KNOWLEDGE_BASE = [
    ('refund',   'Refund Policy. Customers may request a full refund within 30 days of purchase for any '
                 'reason. Refunds are processed to the original payment method within 5 business days. '
                 'There is no restocking fee for standard plans. Enterprise hardware returns are subject '
                 'to a 15% restocking fee.'),
    ('shipping', 'Shipping. Standard shipping is free on orders over $50; orders under $50 incur a flat '
                 '$4.99 fee. Standard delivery takes 3 to 5 business days. Expedited shipping (next '
                 'business day) is available at checkout for an additional charge.'),
    ('warranty', 'Limited Warranty. All hardware is covered by a 12-month limited warranty from the date '
                 'of delivery. The warranty covers manufacturing defects and component failure under '
                 'normal use. It does not cover accidental damage, liquid damage, or unauthorised modification.'),
    ('plans',    'Our Pro plan includes unlimited projects, priority support, and a 30-day free trial. '
                 'Contact sales for volume discounts. All plans are billed monthly or annually.'),
]

## 2. A minimal retriever

No embeddings, no dependencies — just lexical term overlap, returning the single best-matching passage. (Swap in a NIM embedding model here for a production retriever; the evaluation below doesn't change.)

In [ ]:
import re

STOP = {'the','a','an','is','are','do','does','how','what','and','to','of','for','on','in','you','your','can','many'}

def terms(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP and len(w) > 2}

def retrieve(question, k=1):
    q = terms(question)
    scored = sorted(KNOWLEDGE_BASE, key=lambda d: len(q & terms(d[1])), reverse=True)
    return [passage for _name, passage in scored[:k]]

## 3. The NIM-powered RAG answer

Retrieve the context, then ask the NIM model to answer **strictly from that context** — quoting figures exactly and abstaining when the answer isn't there. `temperature=0` keeps generation deterministic, so the evaluation is reproducible.

In [ ]:
SYSTEM_PROMPT = (
    'You are a careful assistant. Answer the question using ONLY the provided context. '
    'Quote figures exactly as they appear and do not introduce any number that is not present '
    'there. If the context does not contain the answer, reply exactly: '
    '"That is not in the provided context." Keep the answer to one or two sentences.'
)

# Canned answers used only when no API key is set, so the notebook still runs.
_MOCK = {
    'refund': 'Customers have 30 days to request a refund, and there is no restocking fee on standard plans.',
    'shipping': 'Standard shipping is free on orders over $50, and standard delivery takes 3 to 5 business days.',
    'warranty': 'It is a 12-month limited warranty covering manufacturing defects and component failure.',
    'enterprise': 'That is not in the provided context.',
}
def _mock_answer(question):
    ql = question.lower()
    for key, val in _MOCK.items():
        if key in ql or (key == 'refund' and 'refund' in ql) or (key == 'enterprise' and 'enterprise' in ql):
            return val
    return 'That is not in the provided context.'

def nim_generate(question, context):
    if client is None:
        return _mock_answer(question)
    resp = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=256,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content.strip()

def rag_answer(question):
    context = '\n'.join(retrieve(question))
    return nim_generate(question, context), context

## 4. The gold set

Each case names the question, the **facts a correct answer must contain** (`anchors`), and whether the question is answerable from the knowledge base at all. The last case is deliberately **unanswerable** — the price isn't in the docs — so a faithful agent must abstain.

In [ ]:
GOLD = [
    {'question': 'How many days do customers have to request a refund, and is there a restocking fee?',
     'anchors': ['30 days', 'no restocking fee'], 'answerable': True},
    {'question': 'Is standard shipping free, and how long does delivery usually take?',
     'anchors': ['free', '3 to 5 business days'], 'allowed_new_numbers': ['50', '4.99'], 'answerable': True},
    {'question': 'How long is the warranty, and what does it cover?',
     'anchors': ['12-month', 'manufacturing defects'], 'answerable': True},
    {'question': 'What is the annual price of the Enterprise plan?',
     'answerable': False},
]

## 5. Evaluate with OpenGATE — deterministic, no judge

For each case: run the RAG agent, then call `check_grounding` on the answer and the exact context it was given. The verdict is pure logic — the same result every run, no grader model, no API cost.

In [ ]:
from opengate_grounding import check_grounding

rows, passed = [], 0
for case in GOLD:
    answer, context = rag_answer(case['question'])
    result = check_grounding(
        answer, context,
        question=case['question'],
        anchors=case.get('anchors'),
        allowed_new_numbers=case.get('allowed_new_numbers'),
        answerable=case.get('answerable', True),
    )
    passed += result.grounded
    rows.append((case['question'][:48], result.grounded, '; '.join(result.issues) or '—'))

print(f'{"QUESTION":50} {"GROUNDED":9} ISSUES')
print('-' * 90)
for q, g, issues in rows:
    print(f'{q:50} {("YES" if g else "NO"):9} {issues}')
print('-' * 90)
print(f'{passed}/{len(GOLD)} answers grounded')

## 6. Gate it in CI

In a test suite, `assert_grounded` turns a grounding regression into a failing build — so a model or prompt change that starts fabricating numbers can't ship silently.

In [ ]:
from opengate_grounding import assert_grounded

def test_refund_answer_is_grounded():
    answer, context = rag_answer('How many days do customers have to request a refund, and is there a restocking fee?')
    assert_grounded(answer, context, anchors=['30 days', 'no restocking fee'])

test_refund_answer_is_grounded()
print('grounding gate passed ✓')

## Where to go next

- **DeepEval users:** the same check is a drop-in metric — `pip install "opengate-grounding[deepeval]"` and add `GroundingMetric()` alongside your model-graded metrics. See the [package README](https://pypi.org/project/opengate-grounding/).
- **Full framework + CI gate (Node):** the sibling [`node/`](./node) folder wraps a NIM adapter in the OpenGATE runner so `opengate --adapter ./nim-adapter.mjs --datasets ./datasets --online --ci` fails the build on any grounding regression, with a committed baseline.
- **Run anywhere:** the same deterministic check is also an [MCP server](https://github.com/nickjlamb/opengate/tree/main/mcp) and a [Docker image](https://hub.docker.com/r/pharmatools/opengate).

The point: OpenGATE isn't tied to one model or framework. Point it at a NIM-powered agent, a hosted API, or a local pipeline — the grounding verdict is the same, and there's no second model to trust.